# Process and download GEP data

In [1]:
import os, sys, logging, importlib, math, time, boto3, shutil
import rasterio, affine

import geopandas as gpd
import pandas as pd

from zipfile import ZipFile
from affine import Affine
from rasterio import features
from rasterio.mask import mask
from rasterio.features import rasterize

import GEP
from GEP import gep_results

'''prints the time along with the message'''
def tPrint(s):
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

In [14]:
scenario = "0_0_0_0_1_0" 
base_folder = r'C:\WBG\Work\GEP\Climate-2022'
data_folder = os.path.join(base_folder, 'Data')
results_folder = os.path.join(base_folder, 'Results')
rasterized_results_folder = os.path.join(results_folder, 'Rasterized_Scenarios')
scenario_base_folder =  os.path.join(data_folder, 'Scenarios')
scenario_folder = os.path.join(scenario_base_folder, scenario)
summaries_folder = os.path.join(data_folder, 'Scenario_Summaries')
zip_folder =       os.path.join(scenario_base_folder, 'zipfiles')
spatial_folder =   os.path.join(data_folder, 'SpatialData')


## Download results from S3

In [9]:
bucket = "wbg-geography01"
prefix = 'GEP/GEP-Climate-2022/'
s3client = boto3.client('s3', verify=False)
s3 = boto3.resource('s3')

# Loop through the S3 bucket and get all the keys for files that are .tif 
more_results = True
loops = 0
verbose=True
good_res = []
while more_results:
    if loops > 0:
        objects = s3client.list_objects_v2(Bucket=bucket, Prefix=prefix, ContinuationToken=token)
    else:
        objects = s3client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    more_results = objects['IsTruncated']
    if more_results:
        token = objects['NextContinuationToken']
    loops += 1
    for res in objects['Contents']:
        if res['Key'].endswith('zip'):            
            cur_variable = res['Key'].split("/")[-3]
            good_res.append(res['Key'])
        
    

c:\wbg\Anaconda3\envs\urban_test\Lib\site-packages\urllib3\connectionpool.py:1100: InsecureRequestWarning: Unverified HTTPS request is being made to host 'wbg-geography01.s3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


# Unzip shape data

In [27]:
zip_folder = r"C:\Users\WB411133\Downloads\GEP_feats"
zip_files = [os.path.join(zip_folder, x) for x in os.listdir(zip_folder) if x.endswith('zip')]

for z_file in zip_files:
    out_folder = os.path.join(spatial_folder, os.path.basename(z_file).split(".")[0])
    with ZipFile(z_file, 'r') as zip_ref:
        zip_ref.extractall(out_folder)
    print(out_folder)

C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\ao-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\bd-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\bf-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\bi-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\bj-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\bw-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\cd-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\cf-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\cg-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\ci-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\cm-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\dj-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\er-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\fm-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\ga-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\gh-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\gm-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\gn-1
C:\WBG\Work\GEP\Climate-2022\Data\SpatialData\gq-1
C:\WBG\Work\GEP\Climate-2022\Da

## Extract scenario files from zipfiles

In [4]:
importlib.reload(GEP)
   
for zFile in os.listdir(zip_folder):
    cur_zip = os.path.join(zip_folder, zFile)
    country = "-".join(zFile.split("-")[:2])  
    out_scenario_file = os.path.join(scenario_folder, f'{country}-{scenario}.csv')
    if not os.path.exists(out_scenario_file):
        if country in ['fm-2','sb-2','so-2','vu-2']:
            spl = scenario.split("_")
            new_scenario = "_".join([spl[0],spl[1],spl[3]])
            cur_res = GEP.gep_results(country, scenariosFolder=scenario_base_folder, scenario=scenario)
            out_file = cur_res.extract_sample_scenario(new_scenario, scenario_folder)
            os.rename(out_file, out_scenario_file)        
        else:
            cur_res = GEP.gep_results(country, scenariosFolder=scenario_base_folder, scenario=scenario)
            cur_res.extract_sample_scenario(scenario, scenario_folder)
        print(country)

sb-2
so-2
vu-2


## Connect scenario data to geospatial data

In [16]:
scenario_file = os.path.join(scenario_base_folder, "3-0_0_0_0_1_0_Global_merge.csv")
in_scenario = pd.read_csv(scenario_file)

C:\Users\WB411133\AppData\Local\Temp\ipykernel_22384\3265971469.py:2: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  in_scenario = pd.read_csv(scenario_file)


In [22]:
sel_scenario = in_scenario.loc[:,['id','Country', 'IsUrban','NightLights', 'Pop','ElecPopCalib','FinalElecCode2030']]
sel_scenario

,id,Country,IsUrban,NightLights,Pop,ElecPopCalib,FinalElecCode2030
0,1,Angola,0,0.00000,10.09000,0.000000,3.0
1,2,Angola,0,0.00000,149.47000,0.000000,2.0
2,3,Angola,0,0.00000,10.09000,0.000000,3.0
3,4,Angola,0,0.00000,20.17000,0.000000,3.0
4,5,Angola,0,0.00000,30.26000,0.000000,3.0
...,...,...,...,...,...,...,...
10134471,270749,Zimbabwe,0,0.00000,50.29283,0.000000,2.0
10134472,162,Zimbabwe,0,0.00000,66.68167,0.000000,3.0
10134473,294214,Zimbabwe,0,0.00000,15.54817,0.000000,3.0
10134474,218587,Zimbabwe,0,2.92987,59.84680,62.204357,2.0


In [ ]:
importlib.reload(GEP)

for country in countries:
    tPrint(country)
    out_file = '/media/gost/DATA1/GEP/Country_Scenarios/%s_electrifrication_status.shp' % country
    if not os.path.exists(out_file):
        xx_so = GEP.gep_results(country, scenario=scenario)
        res = xx_so.join_results()
        sel_res = res.loc[:,['FinalElecCode2018','Pop','geometry']]
        sel_res.to_file(out_file)
        